# Phase 3-5: Attack Detection & Classification

Evaluate detection performance and train classifiers to distinguish attack types.

In [ ]:
import sys
import os
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, classification_report, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, train_test_split

sns.set_style("whitegrid")

# colab setup
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = "/content/drive/MyDrive/Colab Notebooks/data"
else:
    BASE = "/Users/tyreecruse/Desktop/CS230/Project/Data"

FEATURES_DIR = f"{BASE}/analysis/yolo_features"
RESULTS_DIR = f"{BASE}/analysis/results"
FIGURES_DIR = f"{BASE}/analysis/figures"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

print(f"Features dir: {FEATURES_DIR}")

In [ ]:
# load clean
with open(f"{FEATURES_DIR}/clean_yolo_features.pkl", 'rb') as f:
    data = pickle.load(f)
cleanFeats = data['features']
print(f"clean: {cleanFeats.shape}")

# centroid
centroid = cleanFeats.mean(axis=0)
centroid = centroid / np.linalg.norm(centroid)
cleanDist = 1 - (cleanFeats @ centroid)
print(f"clean dist mean: {cleanDist.mean():.4f}")

In [ ]:
# detection performance
print("="*60)
print("DETECTION")
print("="*60)

attacks = [
    "fgsm_030", "fgsm_045", "fgsm_060", "fgsm_075", "fgsm_090", "fgsm_105",
    "gaussian_010", "gaussian_050", "gaussian_150", "gaussian_200", "gaussian_250",
    "patches",
]

results = []
rocCurves = {}  # save for plotting

for atk in attacks:
    path = f"{FEATURES_DIR}/{atk}_yolo_features.pkl"
    if not os.path.exists(path):
        print(f"{atk}: not found")
        continue
    
    with open(path, 'rb') as f:
        data = pickle.load(f)
    advFeats = data['features']
    advDist = 1 - (advFeats @ centroid)
    
    # roc
    allDist = np.concatenate([cleanDist, advDist])
    labels = np.concatenate([np.zeros(len(cleanDist)), np.ones(len(advDist))])
    fpr, tpr, _ = roc_curve(labels, allDist)
    rocAuc = auc(fpr, tpr)
    
    # figure out attack type
    if 'fgsm' in atk:
        atype = 'FGSM'
        strength = int(atk.split('_')[1]) / 1000
    elif 'gaussian' in atk:
        atype = 'Gaussian'
        strength = int(atk.split('_')[1]) / 1000
    else:
        atype = 'Patch'
        strength = 0
    
    results.append({'attack': atk, 'type': atype, 'strength': strength, 'auc': rocAuc})
    rocCurves[atk] = (fpr, tpr, rocAuc)
    
    print(f"{atk:15} {atype:10} {strength:.3f}  AUC={rocAuc:.3f}")

print(f"\n{len(results)} attacks processed")

In [ ]:
# plot roc curves
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# fgsm
fgsm_attacks = [r for r in results if r['type'] == 'FGSM']
fgsm_attacks.sort(key=lambda x: x['strength'])
colors = plt.cm.Reds(np.linspace(0.3, 0.9, len(fgsm_attacks)))
for i, r in enumerate(fgsm_attacks):
    fpr, tpr, auc_val = rocCurves[r['attack']]
    axes[0].plot(fpr, tpr, color=colors[i], lw=2, label=f"ε={r['strength']:.3f} ({auc_val:.2f})")
axes[0].plot([0,1], [0,1], 'k--')
axes[0].set_title('FGSM')
axes[0].set_xlabel('FPR')
axes[0].set_ylabel('TPR')
axes[0].legend(fontsize=8)

# gaussian
gauss_attacks = [r for r in results if r['type'] == 'Gaussian']
gauss_attacks.sort(key=lambda x: x['strength'])
colors = plt.cm.Greens(np.linspace(0.3, 0.9, len(gauss_attacks)))
for i, r in enumerate(gauss_attacks):
    fpr, tpr, auc_val = rocCurves[r['attack']]
    axes[1].plot(fpr, tpr, color=colors[i], lw=2, label=f"σ={r['strength']:.3f} ({auc_val:.2f})")
axes[1].plot([0,1], [0,1], 'k--')
axes[1].set_title('Gaussian')
axes[1].set_xlabel('FPR')
axes[1].set_ylabel('TPR')
axes[1].legend(fontsize=8)

# patches
patch_attacks = [r for r in results if r['type'] == 'Patch']
for r in patch_attacks:
    fpr, tpr, auc_val = rocCurves[r['attack']]
    axes[2].plot(fpr, tpr, 'purple', lw=2, label=f"Patch ({auc_val:.2f})")
axes[2].plot([0,1], [0,1], 'k--')
axes[2].set_title('Patches')
axes[2].set_xlabel('FPR')
axes[2].set_ylabel('TPR')
axes[2].legend()

plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/detection_roc.png", dpi=150)
plt.show()

In [ ]:
# build classification dataset
print("="*60)
print("CLASSIFICATION")
print("="*60)

# load samples from each attack type
fgsm_feats = []
for atk in ['fgsm_030', 'fgsm_045', 'fgsm_060', 'fgsm_075', 'fgsm_090', 'fgsm_105']:
    try:
        with open(f"{FEATURES_DIR}/{atk}_yolo_features.pkl", 'rb') as f:
            data = pickle.load(f)
        fgsm_feats.append(data['features'])
    except:
        pass
fgsm_all = np.vstack(fgsm_feats)
print(f"FGSM: {fgsm_all.shape}")

gauss_feats = []
for atk in ['gaussian_010', 'gaussian_050', 'gaussian_150', 'gaussian_200', 'gaussian_250']:
    try:
        with open(f"{FEATURES_DIR}/{atk}_yolo_features.pkl", 'rb') as f:
            data = pickle.load(f)
        gauss_feats.append(data['features'])
    except:
        pass
gauss_all = np.vstack(gauss_feats)
print(f"Gaussian: {gauss_all.shape}")

try:
    with open(f"{FEATURES_DIR}/patches_yolo_features.pkl", 'rb') as f:
        data = pickle.load(f)
    patch_all = data['features']
    print(f"Patch: {patch_all.shape}")
except:
    patch_all = None
    print("Patch: not found")

In [ ]:
# sample 2000 from each and combine
n = 2000

idx = np.random.choice(len(fgsm_all), min(n, len(fgsm_all)), replace=False)
X_fgsm = fgsm_all[idx]
y_fgsm = np.zeros(len(idx))  # 0 = FGSM

idx = np.random.choice(len(gauss_all), min(n, len(gauss_all)), replace=False)
X_gauss = gauss_all[idx]
y_gauss = np.ones(len(idx))  # 1 = Gaussian

if patch_all is not None:
    idx = np.random.choice(len(patch_all), min(n, len(patch_all)), replace=False)
    X_patch = patch_all[idx]
    y_patch = np.full(len(idx), 2)  # 2 = Patch
    X = np.vstack([X_fgsm, X_gauss, X_patch])
    y = np.concatenate([y_fgsm, y_gauss, y_patch])
else:
    X = np.vstack([X_fgsm, X_gauss])
    y = np.concatenate([y_fgsm, y_gauss])

print(f"Total samples: {len(X)}")
print(f"Classes: {np.unique(y)}")

# split
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {len(Xtrain)}, Test: {len(Xtest)}")

In [ ]:
# try different classifiers
print("\nKNN k=5...")
knn5 = KNeighborsClassifier(n_neighbors=5)
scores = cross_val_score(knn5, Xtrain, ytrain, cv=5)
print(f"  CV: {scores.mean():.3f} +/- {scores.std()*2:.3f}")
knn5.fit(Xtrain, ytrain)
acc_knn5 = knn5.score(Xtest, ytest)
print(f"  Test: {acc_knn5:.3f}")

print("\nKNN k=10...")
knn10 = KNeighborsClassifier(n_neighbors=10)
scores = cross_val_score(knn10, Xtrain, ytrain, cv=5)
print(f"  CV: {scores.mean():.3f} +/- {scores.std()*2:.3f}")
knn10.fit(Xtrain, ytrain)
acc_knn10 = knn10.score(Xtest, ytest)
print(f"  Test: {acc_knn10:.3f}")

print("\nRandom Forest...")
rf = RandomForestClassifier(n_estimators=100, random_state=42)
scores = cross_val_score(rf, Xtrain, ytrain, cv=5)
print(f"  CV: {scores.mean():.3f} +/- {scores.std()*2:.3f}")
rf.fit(Xtrain, ytrain)
acc_rf = rf.score(Xtest, ytest)
print(f"  Test: {acc_rf:.3f}")

# best
best_acc = max(acc_knn5, acc_knn10, acc_rf)
if best_acc == acc_rf:
    best_clf = rf
    best_name = 'Random Forest'
elif best_acc == acc_knn10:
    best_clf = knn10
    best_name = 'KNN k=10'
else:
    best_clf = knn5
    best_name = 'KNN k=5'

print(f"\nBest: {best_name} ({best_acc:.1%})")

In [ ]:
# confusion matrix
ypred = best_clf.predict(Xtest)
cm = confusion_matrix(ytest, ypred)

class_names = ['FGSM', 'Gaussian', 'Patch'] if patch_all is not None else ['FGSM', 'Gaussian']

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title(f'Confusion Matrix - {best_name}')
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/classification_cm.png", dpi=150)
plt.show()

print(classification_report(ytest, ypred, target_names=class_names))

In [ ]:
# attack type centroids
print("="*60)
print("CENTROIDS")
print("="*60)

centroids = {
    'Clean': cleanFeats.mean(axis=0),
    'FGSM': fgsm_all.mean(axis=0),
    'Gaussian': gauss_all.mean(axis=0),
}
if patch_all is not None:
    centroids['Patch'] = patch_all.mean(axis=0)

# distances between centroids
names = list(centroids.keys())
print("\nCentroid distances:")
for i in range(len(names)):
    for j in range(i+1, len(names)):
        c1 = centroids[names[i]] / np.linalg.norm(centroids[names[i]])
        c2 = centroids[names[j]] / np.linalg.norm(centroids[names[j]])
        dist = 1 - np.dot(c1, c2)
        print(f"  {names[i]} <-> {names[j]}: {dist:.4f}")

In [ ]:
# distance matrix heatmap
n = len(names)
dist_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        c1 = centroids[names[i]] / np.linalg.norm(centroids[names[i]])
        c2 = centroids[names[j]] / np.linalg.norm(centroids[names[j]])
        dist_matrix[i,j] = 1 - np.dot(c1, c2)

plt.figure(figsize=(8, 6))
sns.heatmap(dist_matrix, annot=True, fmt='.4f', cmap='YlOrRd',
            xticklabels=names, yticklabels=names)
plt.title('Centroid Distances')
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/centroid_distances.png", dpi=150)
plt.show()

In [ ]:
# save results
pd.DataFrame(results).to_csv(f"{RESULTS_DIR}/detection_auc.csv", index=False)
print("saved detection_auc.csv")

clf_results = [
    {'name': 'KNN k=5', 'test_acc': acc_knn5},
    {'name': 'KNN k=10', 'test_acc': acc_knn10},
    {'name': 'Random Forest', 'test_acc': acc_rf},
]
pd.DataFrame(clf_results).to_csv(f"{RESULTS_DIR}/classification_results.csv", index=False)
print("saved classification_results.csv")

In [ ]:
print("="*60)
print("SUMMARY")
print("="*60)

print(f"\nDetection:")
print(f"  Avg AUC: {np.mean([r['auc'] for r in results]):.3f}")
print(f"  Best: {max(results, key=lambda x: x['auc'])['attack']}")
print(f"  Worst: {min(results, key=lambda x: x['auc'])['attack']}")

print(f"\nClassification:")
print(f"  Best: {best_name} ({best_acc:.1%})")
if best_acc > 0.9:
    print("  -> Attack types have distinct signatures")
else:
    print("  -> Some overlap between signatures")